# 03 — Experiment 1: Geometric Alignment

**Research question:**
  Do safety neurons geometrically explain the refusal direction?

This is the first original experiment of your thesis — the first place
where the two papers actually meet. We ask:

  If we take the output vectors of the top safety neurons
  (the directions they write into the residual stream),
  do those vectors collectively span the refusal direction r?

Three measurements:

  1. Per-neuron alignment
     For each safety neuron, compute cosine similarity between its
     output vector and r. Which neurons point most toward r?

  2. Reconstruction curve
     For increasing k, measure what fraction of r is explained by
     the top-k safety neurons' output vectors.
     Compare to a random-neuron baseline.

  3. Layer-wise analysis
     At which layers do safety neurons best explain r?
     Does the geometric alignment match where r is strongest?

**Possible outcomes and their meaning:**

  High variance explained (>0.7) → safety neurons collectively write r
    into the residual stream. The two papers describe the same mechanism
    at different granularities: neurons are the implementation, r is the effect.

  Moderate (0.3–0.7) → partial overlap. Safety neurons contribute to r
    but other sources (attention heads, other neurons) also matter.

  Low (<0.3) → the mechanisms are geometrically dissociable.
    r exists in the residual stream but is not primarily written by
    the neurons that change most during alignment.

All three outcomes are thesis-worthy — the contribution is the measurement.

**Expected runtime:** ~15 minutes
  (no generation, just weight lookups and matrix operations)

## 0. Imports

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/from-neurons-to-directions/src")

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from model_utils import load_model_and_tokenizer, get_num_layers
from safety_neurons import get_top_safety_neurons
from metrics import (
    neuron_output_vectors,
    neuron_contribution_to_direction,
    top_neuron_reconstruction,
    variance_explained,
    cosine_similarity_1d,
)
from viz import (
    plot_reconstruction_curve,
    plot_layer_alignment,
)

print("Imports OK")

## 1. Load saved results from notebooks 01 and 02

We need:
  - directions        : refusal direction at every layer (from nb 01)
  - best_direction    : the single strongest layer's direction (from nb 01)
  - change_scores     : per-neuron alignment change scores (from nb 02)
  - instruct model    : to read weight matrices (W_down per layer)

We do NOT need to run any forward passes in this notebook —
everything is derived from saved tensors and model weights.

In [ ]:
print("Loading saved results...")

directions = torch.load("/kaggle/working/results/directions.pt", map_location="cpu")
best_data  = torch.load("/kaggle/working/results/best_direction.pt", map_location="cpu")
scores     = torch.load("/kaggle/working/results/change_scores.pt",  map_location="cpu")

best_layer = best_data["layer"]
best_r     = best_data["direction"]      # unit vector [hidden_size]

print(f"Loaded directions for {len(directions)} layers")
print(f"Best layer : {best_layer}")
print(f"Best r     : shape={best_r.shape}, norm={best_r.norm():.4f}")

## 2. Load the instruct model (weights only)

We need the instruct model's W_down matrices to compute which direction
each neuron writes into the residual stream.

We only access model.model.layers[l].mlp.down_proj.weight — no forward
passes are needed, so this is fast and memory-light.

In [ ]:
instruct_model, instruct_tokenizer = load_model_and_tokenizer("instruct")
n_layers = get_num_layers(instruct_model)
print(f"Instruct model loaded | {n_layers} layers")

## 3. Select safety neurons at multiple k values

We'll need neuron sets of different sizes for the reconstruction curve.
Compute them all up front.

In [ ]:
k_values  = [10, 25, 50, 100, 250, 500]
neuron_sets = {k: get_top_safety_neurons(scores, k=k) for k in k_values}

print(f"\nNeuron sets prepared: {list(neuron_sets.keys())}")

## 4. Per-neuron alignment with the refusal direction

For every safety neuron in the top-500, measure the cosine similarity
between:
  - the direction that neuron writes into the residual stream (W_down column)
  - the refusal direction r at the best layer

This tells us which specific neurons are geometrically aligned with r.

Note: we use best_r (the refusal direction at the best layer) as our
reference. You can repeat this for other layers if you want a full picture.

In [ ]:
top_500 = neuron_sets[500]

print(f"Computing per-neuron alignment with r at layer {best_layer}...")
contributions = neuron_contribution_to_direction(
    model=instruct_model,
    refusal_direction=best_r,
    safety_neurons=top_500,
)

# Sort by absolute alignment (neurons can be positively or negatively aligned)
ranked = sorted(contributions.items(), key=lambda x: abs(x[1]), reverse=True)

print(f"\nTop-20 neurons most aligned with the refusal direction:")
print(f"{'Rank':<6} {'Layer':<8} {'Neuron':<10} {'Cosine sim':>12}")
print("-" * 40)
for rank, ((layer_idx, neuron_idx), sim) in enumerate(ranked[:20], 1):
    print(f"{rank:<6} {layer_idx:<8} {neuron_idx:<10} {sim:>12.4f}")

# Distribution summary
sims = [abs(v) for v in contributions.values()]
sims_t = torch.tensor(sims)
print(f"\nAlignment distribution (|cosine sim|):")
print(f"  Mean   : {sims_t.mean():.4f}")
print(f"  Median : {sims_t.median():.4f}")
print(f"  Max    : {sims_t.max():.4f}")
print(f"  >0.1   : {(sims_t > 0.1).sum().item()} neurons")
print(f"  >0.3   : {(sims_t > 0.3).sum().item()} neurons")

# Save for thesis table
torch.save(contributions, "/kaggle/working/results/neuron_contributions.pt")
print("\nSaved → results/neuron_contributions.pt")

## 5. Reconstruction curve — safety neurons vs random baseline

This is the central plot of Experiment 1.

For k = [10, 25, 50, 100, 250, 500]:
  Collect the output vectors of the top-k safety neurons → [k, hidden_size]
  Measure what fraction of r lies in the subspace they span

Then repeat with k randomly selected neurons to establish a baseline.

Interpretation:
  Steep rise above the random baseline → safety neurons are specifically
  aligned with r (not just a consequence of having more vectors)

In [ ]:
print("Computing reconstruction curve for SAFETY neurons...")
safety_curve = top_neuron_reconstruction(
    model=instruct_model,
    refusal_direction=best_r,
    safety_neurons=top_500,
    ks=k_values,
)

print("\nComputing reconstruction curve for RANDOM neurons (baseline)...")
import random
random.seed(42)
from model_utils import get_intermediate_size
intermediate_size = get_intermediate_size(instruct_model)

random_neurons_500 = [
    (random.randint(0, n_layers - 1), random.randint(0, intermediate_size - 1))
    for _ in range(500)
]

random_curve = top_neuron_reconstruction(
    model=instruct_model,
    refusal_direction=best_r,
    safety_neurons=random_neurons_500,   # ranked list, first k used
    ks=k_values,
)

print("\nReconstruction results:")
print(f"{'k':<8} {'Safety':>10} {'Random':>10} {'Lift':>10}")
print("-" * 42)
for k in k_values:
    lift = safety_curve[k] - random_curve[k]
    print(f"{k:<8} {safety_curve[k]:>10.4f} {random_curve[k]:>10.4f} {lift:>+10.4f}")

fig = plot_reconstruction_curve(
    curve=safety_curve,
    random_curve=random_curve,
    title="Variance of Refusal Direction Explained by Safety Neurons",
    save_path="/kaggle/working/results/figures/exp1_reconstruction_curve.png",
)
plt.show()

torch.save({"safety": safety_curve, "random": random_curve},
           "/kaggle/working/results/reconstruction_curves.pt")
print("Saved → results/reconstruction_curves.pt")

## 6. Layer-wise reconstruction

Does the geometric alignment vary by layer?

For each layer l, we:
  - Take the refusal direction r_l at that layer
  - Take the safety neurons that live in layer l
  - Measure variance of r_l explained by those neurons

This tells us whether certain layers are "hubs" where safety neurons
and the refusal direction coincide, vs layers where they are dissociated.

In [ ]:
print("Computing layer-wise reconstruction...")

layerwise_results = {}

for layer_idx in range(n_layers):
    if layer_idx not in directions:
        continue

    r_l = directions[layer_idx]   # refusal direction at this layer

    # Safety neurons in this layer only
    layer_neurons = [
        (l, n) for l, n in top_500 if l == layer_idx
    ]

    if len(layer_neurons) == 0:
        layerwise_results[layer_idx] = 0.0
        continue

    # Get their output vectors
    neuron_indices = [n for _, n in layer_neurons]
    vecs = neuron_output_vectors(instruct_model, layer_idx, neuron_indices)

    ve = variance_explained(r_l, vecs)
    layerwise_results[layer_idx] = ve

# Plot as a bar chart using plot_layer_alignment (repurposing for layer-wise VE)
fig, ax = plt.subplots(figsize=(12, 4))
layers = sorted(layerwise_results.keys())
values = [layerwise_results[l] for l in layers]
ax.bar(layers, values, color="#4878CF", alpha=0.8, width=0.7)
ax.axvline(best_layer, color="#D65F5F", linewidth=2, linestyle="--",
           label=f"Best refusal layer ({best_layer})")
ax.set_xlabel("Layer")
ax.set_ylabel("Variance explained in r_l")
ax.set_title("Layer-wise: Safety Neurons Explaining Their Layer's Refusal Direction")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/results/figures/exp1_layerwise.png", dpi=150)
plt.show()

torch.save(layerwise_results, "/kaggle/working/results/layerwise_reconstruction.pt")
print("Saved → results/layerwise_reconstruction.pt")

## 7. Cross-layer analysis

So far we've kept things layer-local: r_l vs neurons in layer l.
But safety neurons in layer l write into the residual stream, which
then flows into layer l+1, l+2, etc.

Here we ask: do safety neurons from ALL layers together better explain
the refusal direction at the best layer?

If cross-layer neurons explain more than within-layer neurons alone,
it suggests refusal is built up across multiple layers rather than
implemented in one place.

In [ ]:
print(f"Cross-layer analysis: all safety neurons → r at layer {best_layer}")

# All 500 safety neurons, regardless of which layer they're in
all_vecs = []
by_layer = {}
for layer_idx, neuron_idx in top_500:
    by_layer.setdefault(layer_idx, []).append(neuron_idx)

for layer_idx, neuron_indices in by_layer.items():
    vecs = neuron_output_vectors(instruct_model, layer_idx, neuron_indices)
    all_vecs.append(vecs)

all_vecs_cat = torch.cat(all_vecs, dim=0)   # [500, hidden_size]
ve_cross = variance_explained(best_r, all_vecs_cat)

# Compare to within-layer only (neurons at best_layer)
best_layer_neurons = [n for l, n in top_500 if l == best_layer]
if best_layer_neurons:
    best_layer_vecs = neuron_output_vectors(instruct_model, best_layer, best_layer_neurons)
    ve_within = variance_explained(best_r, best_layer_vecs)
else:
    ve_within = 0.0

print(f"\nVariance explained in r (layer {best_layer}):")
print(f"  Within-layer neurons only : {ve_within:.4f} ({ve_within:.1%})")
print(f"  All-layer neurons         : {ve_cross:.4f} ({ve_cross:.1%})")
print(f"  Cross-layer gain          : {ve_cross - ve_within:+.4f}")

## 8. Summary and thesis interpretation

Print a structured summary you can copy directly into your thesis notes.

In [ ]:
print("=" * 60)
print("EXPERIMENT 1 — GEOMETRIC ALIGNMENT SUMMARY")
print("=" * 60)
print(f"\nRefusal direction r at layer {best_layer}")
print(f"\nPer-neuron alignment (top-500 safety neurons):")
print(f"  Mean |cosine sim| : {sims_t.mean():.4f}")
print(f"  Max  |cosine sim| : {sims_t.max():.4f}")
print(f"  Neurons with |sim| > 0.3 : {(sims_t > 0.3).sum().item()}")

print(f"\nReconstruction curve (variance explained in r):")
for k in k_values:
    print(f"  k={k:<5}: safety={safety_curve[k]:.3f}  random={random_curve[k]:.3f}  "
          f"lift={safety_curve[k]-random_curve[k]:+.3f}")

print(f"\nCross-layer analysis:")
print(f"  Within best layer : {ve_within:.3f}")
print(f"  All layers        : {ve_cross:.3f}")

print(f"\nInterpretation:")
if safety_curve[500] > 0.7:
    print("  HIGH alignment — safety neurons substantially implement r.")
    print("  The two papers describe the same mechanism at different levels.")
elif safety_curve[500] > 0.3:
    print("  MODERATE alignment — safety neurons partially implement r.")
    print("  Other sources (attention, other neurons) also contribute.")
else:
    print("  LOW alignment — mechanisms are geometrically dissociable.")
    print("  r exists but is not primarily written by safety neurons.")
print("=" * 60)
print("\nFiles saved:")
print("  results/neuron_contributions.pt")
print("  results/reconstruction_curves.pt")
print("  results/layerwise_reconstruction.pt")
print("  results/figures/exp1_reconstruction_curve.png")
print("  results/figures/exp1_layerwise.png")
print("\nNext: run 04_exp2_causal_bridge.ipynb")